In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f4feb79f",
   "metadata": {
    "vscode": {
     "languageId": "plaintext"
    }
   },
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import yfinance as yf\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from statsmodels.tsa.stattools import adfuller\n",
    "from scipy import stats\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.float_format', lambda x: '%.4f' % x)\n",
    "\n",
    "print(\"Libraries imported successfully!\")\n",
    "\n",
    "\n",
    "# Define assets and period\n",
    "tickers = ['TSLA', 'BND', 'SPY']\n",
    "start_date = '2015-01-01'\n",
    "end_date = '2026-06-30'\n",
    "\n",
    "print(\"Downloading data from YFinance...\")\n",
    "print(f\"Tickers: {tickers}\")\n",
    "print(f\"Period: {start_date} to {end_date}\")\n",
    "\n",
    "# Download data\n",
    "data = yf.download(tickers, start=start_date, end=end_date)\n",
    "\n",
    "# Display basic info\n",
    "print(f\"\\nData shape: {data.shape}\")\n",
    "print(f\"Data columns: {data.columns.tolist()}\")\n",
    "print(\"\\nFirst 5 rows:\")\n",
    "print(data.head())\n",
    "\n",
    "# Extract adjusted closing prices\n",
    "closing_prices = data['Adj Close']\n",
    "print(f\"\\nClosing prices shape: {closing_prices.shape}\")\n",
    "print(closing_prices.head())\n",
    "\n",
    "# Check for missing values\n",
    "print(\"Missing values before cleaning:\")\n",
    "print(closing_prices.isnull().sum())\n",
    "\n",
    "# Handle missing values - forward fill then backward fill\n",
    "closing_prices_clean = closing_prices.fillna(method='ffill').fillna(method='bfill')\n",
    "\n",
    "print(\"\\nMissing values after cleaning:\")\n",
    "print(closing_prices_clean.isnull().sum())\n",
    "\n",
    "# Check data types\n",
    "print(\"\\nData types:\")\n",
    "print(closing_prices_clean.dtypes)\n",
    "\n",
    "# Calculate daily returns\n",
    "returns_clean = closing_prices_clean.pct_change().dropna()\n",
    "print(f\"\\nReturns shape: {returns_clean.shape}\")\n",
    "print(\"Returns head:\")\n",
    "print(returns_clean.head())\n",
    "\n",
    "# Basic statistics\n",
    "print(\"\\nClosing prices statistics:\")\n",
    "print(closing_prices_clean.describe())\n",
    "\n",
    "# Figure 1: Closing Prices Over Time\n",
    "fig, axes = plt.subplots(3, 1, figsize=(15, 12))\n",
    "\n",
    "for i, ticker in enumerate(tickers):\n",
    "    ax = axes[i]\n",
    "    ax.plot(closing_prices_clean.index, closing_prices_clean[ticker], \n",
    "            label=f'{ticker} Price', linewidth=2, color=['blue', 'green', 'red'][i])\n",
    "    ax.set_title(f'{ticker} - Closing Price (2015-2026)', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('Date')\n",
    "    ax.set_ylabel('Price ($)')\n",
    "    ax.legend()\n",
    "    ax.grid(True, alpha=0.3)\n",
    "    \n",
    "    # Add horizontal line at mean\n",
    "    mean_price = closing_prices_clean[ticker].mean()\n",
    "    ax.axhline(y=mean_price, color='black', linestyle='--', alpha=0.5, \n",
    "               label=f'Mean: ${mean_price:.2f}')\n",
    "    ax.legend()\n",
    "\n",
    "plt.suptitle('Historical Stock Prices', fontsize=16, fontweight='bold', y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/processed/closing_prices.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "# Figure 2: Daily Returns Distribution with Statistics\n",
    "fig, axes = plt.subplots(3, 1, figsize=(15, 12))\n",
    "\n",
    "for i, ticker in enumerate(tickers):\n",
    "    ax = axes[i]\n",
    "    returns = returns_clean[ticker]\n",
    "    \n",
    "    # Histogram\n",
    "    n, bins, patches = ax.hist(returns, bins=50, alpha=0.7, edgecolor='black', \n",
    "                                color=['blue', 'green', 'red'][i])\n",
    "    \n",
    "    # Add vertical lines\n",
    "    ax.axvline(returns.mean(), color='red', linestyle='--', linewidth=2, \n",
    "               label=f'Mean: {returns.mean():.4f}')\n",
    "    ax.axvline(returns.median(), color='orange', linestyle='--', linewidth=2, \n",
    "               label=f'Median: {returns.median():.4f}')\n",
    "    ax.axvline(0, color='black', linestyle='-', alpha=0.5)\n",
    "    \n",
    "    # Add text with statistics\n",
    "    stats_text = f'Skewness: {returns.skew():.3f}\\nKurtosis: {returns.kurtosis():.3f}'\n",
    "    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, \n",
    "            verticalalignment='top', horizontalalignment='right',\n",
    "            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))\n",
    "    \n",
    "    ax.set_title(f'{ticker} - Daily Returns Distribution', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('Daily Return')\n",
    "    ax.set_ylabel('Frequency')\n",
    "    ax.legend()\n",
    "    ax.grid(True, alpha=0.3)\n",
    "\n",
    "plt.suptitle('Daily Returns Distribution', fontsize=16, fontweight='bold', y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/processed/returns_distribution.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "# Figure 3: Rolling Volatility (30-day and 90-day)\n",
    "fig, axes = plt.subplots(3, 1, figsize=(15, 12))\n",
    "\n",
    "for i, ticker in enumerate(tickers):\n",
    "    ax = axes[i]\n",
    "    returns = returns_clean[ticker]\n",
    "    \n",
    "    rolling_std_30 = returns.rolling(window=30).std()\n",
    "    rolling_std_90 = returns.rolling(window=90).std()\n",
    "    \n",
    "    ax.plot(returns.index, rolling_std_30, label='30-day Volatility', \n",
    "            linewidth=2, color='blue')\n",
    "    ax.plot(returns.index, rolling_std_90, label='90-day Volatility', \n",
    "            linewidth=2, color='red', alpha=0.7)\n",
    "    \n",
    "    ax.set_title(f'{ticker} - Rolling Volatility', fontsize=14, fontweight='bold')\n",
    "    ax.set_xlabel('Date')\n",
    "    ax.set_ylabel('Volatility')\n",
    "    ax.legend()\n",
    "    ax.grid(True, alpha=0.3)\n",
    "\n",
    "plt.suptitle('Rolling Volatility Analysis', fontsize=16, fontweight='bold', y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/processed/rolling_volatility.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "# Figure 4: Correlation Matrix\n",
    "correlation_matrix = returns_clean.corr()\n",
    "\n",
    "plt.figure(figsize=(10, 8))\n",
    "sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,\n",
    "            square=True, linewidths=2, fmt='.3f',\n",
    "            cbar_kws={\"shrink\": 0.8})\n",
    "plt.title('Asset Returns Correlation Matrix', fontsize=16, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/processed/correlation_matrix.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\nCorrelation Matrix:\")\n",
    "print(correlation_matrix)\n",
    "\n",
    "\n",
    "# Augmented Dickey-Fuller Test\n",
    "def adf_test(series, series_name, significance=0.05):\n",
    "    \"\"\"Perform ADF test and interpret results\"\"\"\n",
    "    print(f\"\\n{'='*60}\")\n",
    "    print(f\"ADF Test: {series_name}\")\n",
    "    print('='*60)\n",
    "    \n",
    "    result = adfuller(series.dropna(), autolag='AIC')\n",
    "    \n",
    "    print(f'Test Statistic: {result[0]:.6f}')\n",
    "    print(f'p-value: {result[1]:.6f}')\n",
    "    print('Critical Values:')\n",
    "    for key, value in result[4].items():\n",
    "        print(f'  {key}: {value:.6f}')\n",
    "    \n",
    "    if result[1] <= significance:\n",
    "        print(f'\\n✓ Result: Reject H0 - {series_name} is STATIONARY')\n",
    "        return True\n",
    "    else:\n",
    "        print(f'\\n✗ Result: Fail to reject H0 - {series_name} is NON-STATIONARY')\n",
    "        return False\n",
    "\n",
    "# Test closing prices\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"TESTING CLOSING PRICES\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "stationary_results = {}\n",
    "for ticker in tickers:\n",
    "    is_stationary = adf_test(closing_prices_clean[ticker], f\"{ticker} Closing Price\")\n",
    "    stationary_results[f\"{ticker}_price\"] = is_stationary\n",
    "\n",
    "# Test daily returns\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"TESTING DAILY RETURNS\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "for ticker in tickers:\n",
    "    is_stationary = adf_test(returns_clean[ticker], f\"{ticker} Daily Returns\")\n",
    "    stationary_results[f\"{ticker}_returns\"] = is_stationary\n",
    "\n",
    "# Summary\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"STATIONARITY SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "for key, value in stationary_results.items():\n",
    "    status = \"Stationary\" if value else \"Non-Stationary\"\n",
    "    print(f\"{key}: {status}\")\n",
    "\n",
    "    # Calculate Value at Risk (VaR)\n",
    "def calculate_var(returns, confidence_level=0.95):\n",
    "    \"\"\"Calculate Value at Risk\"\"\"\n",
    "    return np.percentile(returns, (1 - confidence_level) * 100)\n",
    "\n",
    "def calculate_cvar(returns, confidence_level=0.95):\n",
    "    \"\"\"Calculate Conditional VaR (Expected Shortfall)\"\"\"\n",
    "    var = calculate_var(returns, confidence_level)\n",
    "    return returns[returns <= var].mean()\n",
    "\n",
    "def calculate_sharpe_ratio(returns, risk_free_rate=0.02):\n",
    "    \"\"\"Calculate annualized Sharpe Ratio\"\"\"\n",
    "    excess_returns = returns - risk_free_rate/252\n",
    "    return np.sqrt(252) * excess_returns.mean() / returns.std()\n",
    "\n",
    "def calculate_sortino_ratio(returns, risk_free_rate=0.02, target_return=0):\n",
    "    \"\"\"Calculate Sortino Ratio\"\"\"\n",
    "    downside_returns = returns[returns < target_return]\n",
    "    excess_return = returns.mean() - risk_free_rate/252\n",
    "    downside_deviation = downside_returns.std()\n",
    "    return np.sqrt(252) * excess_return / downside_deviation if downside_deviation != 0 else np.nan\n",
    "\n",
    "# Calculate metrics\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"RISK METRICS\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "risk_metrics = {}\n",
    "\n",
    "for ticker in tickers:\n",
    "    returns = returns_clean[ticker]\n",
    "    \n",
    "    metrics = {\n",
    "        '95% VaR': calculate_var(returns, 0.95),\n",
    "        '99% VaR': calculate_var(returns, 0.99),\n",
    "        '95% CVaR': calculate_cvar(returns, 0.95),\n",
    "        'Sharpe Ratio': calculate_sharpe_ratio(returns),\n",
    "        'Sortino Ratio': calculate_sortino_ratio(returns),\n",
    "        'Max Drawdown': (returns.cumsum().cummax() - returns.cumsum()).min(),\n",
    "        'Skewness': returns.skew(),\n",
    "        'Kurtosis': returns.kurtosis()\n",
    "    }\n",
    "    \n",
    "    risk_metrics[ticker] = metrics\n",
    "    \n",
    "    print(f\"\\n{ticker}:\")\n",
    "    for key, value in metrics.items():\n",
    "        print(f\"  {key}: {value:.4f}\")\n",
    "\n",
    "# Create summary DataFrame\n",
    "risk_df = pd.DataFrame(risk_metrics).T\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"RISK METRICS SUMMARY TABLE\")\n",
    "print(\"=\"*60)\n",
    "print(risk_df.round(4))\n",
    "\n",
    "# Detect outliers using IQR method\n",
    "def detect_outliers_iqr(data):\n",
    "    \"\"\"Detect outliers using IQR method\"\"\"\n",
    "    Q1 = data.quantile(0.25)\n",
    "    Q3 = data.quantile(0.75)\n",
    "    IQR = Q3 - Q1\n",
    "    lower_bound = Q1 - 1.5 * IQR\n",
    "    upper_bound = Q3 + 1.5 * IQR\n",
    "    outliers = data[(data < lower_bound) | (data > upper_bound)]\n",
    "    return outliers\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"OUTLIER DETECTION\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "for ticker in tickers:\n",
    "    returns = returns_clean[ticker]\n",
    "    outliers = detect_outliers_iqr(returns)\n",
    "    \n",
    "    print(f\"\\n{ticker}:\")\n",
    "    print(f\"  Total outliers: {len(outliers)}\")\n",
    "    print(f\"  Percentage outliers: {len(outliers)/len(returns)*100:.2f}%\")\n",
    "    \n",
    "    if len(outliers) > 0:\n",
    "        print(\"  Top 5 outliers:\")\n",
    "        print(outliers.nlargest(5))\n",
    "        print(\"  Bottom 5 outliers:\")\n",
    "        print(outliers.nsmallest(5))\n",
    "\n",
    "        # Save processed data\n",
    "print(\"\\nSaving processed data...\")\n",
    "\n",
    "# Save closing prices\n",
    "closing_prices_clean.to_csv('../data/processed/closing_prices.csv')\n",
    "print(f\"Closing prices saved to: ../data/processed/closing_prices.csv\")\n",
    "\n",
    "# Save daily returns\n",
    "returns_clean.to_csv('../data/processed/daily_returns.csv')\n",
    "print(f\"Daily returns saved to: ../data/processed/daily_returns.csv\")\n",
    "\n",
    "# Save risk metrics\n",
    "risk_df.to_csv('../data/processed/risk_metrics.csv')\n",
    "print(f\"Risk metrics saved to: ../data/processed/risk_metrics.csv\")\n",
    "\n",
    "# Create summary report\n",
    "summary_stats = {\n",
    "    'Data Period': [f'{start_date} to {end_date}'],\n",
    "    'Number of Days': [len(closing_prices_clean)],\n",
    "    'Assets': [', '.join(tickers)],\n",
    "    'TSLA Mean Return': [f\"{returns_clean['TSLA'].mean():.4%}\"],\n",
    "    'BND Mean\n",
    "     Return': [f\"{returns_clean['BND'].mean():.4%}\"],\n",
    "    'SPY Mean Return': [f\"{returns_clean['SPY'].mean():.4%}\"],\n",
    "    'TSLA Volatility': [f\"{returns_clean['TSLA'].std():.4%}\"],\n",
    "    'BND Volatility': [f\"{returns_clean['BND'].std():.4%}\"],\n",
    "    'SPY Volatility': [f\"{returns_clean['SPY'].std():.4%}\"],\n",
    "    'TSLA Max Drawdown': [f\"{risk_df.loc['TSLA', 'Max Drawdown']:.2%}\"],\n",
    "    'BND Max Drawdown': [f\"{risk_df.loc['BND', 'Max Drawdown']:.2%}\"],\n",
    "    'SPY Max Drawdown': [f\"{risk_df.loc['SPY', 'Max Drawdown']:.2%}\"]\n",
    "}\n",
    "\n",
    "summary_df = pd.DataFrame(summary_stats)\n",
    "summary_df.to_csv('../data/processed/eda_summary.csv', index=False)\n",
    "print(f\"EDA summary saved to: ../data/processed/eda_summary.csv\")\n",
    "\n",
    "print(\"\\n✓ Task 1 completed successfully!\")\n",
    "print(f\"Data period: {start_date} to {end_date}\")\n",
    "print(f\"Total days: {len(closing_prices_clean)}\")"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}